# X-ray threat classification: full-data run on a Colab T4

Runs the same scripts as the local CPU run, on the **full** pair-grouped split (`splits/train.csv`, `val.csv`, `test.csv`).
The split CSVs are part of the repo, so Colab uses exactly the same images per split as the local run. Nothing is re-drawn.

1. Runtime -> Change runtime type -> **T4 GPU**.
2. Put this project folder (code + `splits/`, without `xray_baggage/`) on GitHub or Google Drive, then set `PROJECT` below.
3. Run all cells. `evaluate.py` touches the test set once and refuses to re-run it afterwards.

Data licence: DvXray and SIXray are academic use only. Do not copy the images back into the repo.

In [ ]:
!nvidia-smi
import torch; assert torch.cuda.is_available(), 'Select a GPU runtime first'

In [ ]:
# Option A: clone your repo   Option B: mount Drive and cd into the uploaded folder
# !git clone https://github.com/<you>/xray-threat-classification.git
# from google.colab import drive; drive.mount('/content/drive')
PROJECT = '/content/xray-threat-classification'
%cd {PROJECT}
!ls splits

In [ ]:
# Colab already ships CUDA torch/torchvision; only add what is missing
!pip -q install imagehash "huggingface_hub>=0.34" 

In [ ]:
!hf download FrenchCastle/xray-baggages-customs --repo-type dataset --local-dir xray_baggage    --include "annotations/dvxray_coco.json" "images_dvxray.zip" "annotations/sixray_yolo_subset_coco.json" "images_sixray_yolo_subset.zip" MANIFEST.csv
!cd xray_baggage && unzip -q images_dvxray.zip && unzip -q images_sixray_yolo_subset.zip && rm images_*.zip

In [ ]:
# Sanity check: every image named in the committed split CSVs exists, and the leak checks still hold
import pandas as pd, os
s = {k: pd.read_csv(f'splits/{k}.csv') for k in ['train', 'val', 'test']}
for k, d in s.items():
    missing = [f for f in d.file_name if not os.path.exists('xray_baggage/' + f)]
    print(k, len(d), 'dangerous', d.label.sum(), 'missing', len(missing)); assert not missing
for a in s:
    for b in s:
        if a < b:
            assert not set(s[a].bag) & set(s[b].bag) and not set(s[a].phash) & set(s[b].phash)
print('no bag or exact-pHash overlap across splits')

In [ ]:
# Same seed, split, epoch budget and batch size for both models
!python train.py --model baseline --full --epochs 4 --batch-size 32 --workers 2
!python train.py --model tuned    --full --epochs 4 --batch-size 32 --workers 2

In [ ]:
!python evaluate.py --full --workers 2
!python cross_scanner.py --full --workers 2

In [ ]:
from IPython.display import Image, display
display(Image('results/confusion_matrix_full.png')); display(Image('results/roc_full.png'))
# Download results/ (metrics.json, train_*_full.json, test_preds_*_full.csv, plots). Do not download images.
!zip -qr results_full.zip results && ls -la results_full.zip